# Mythos — Interactive Demo

End-to-end walkthrough of the Mythos language model:

1. Load weights from HuggingFace Hub (`bgraudt/mythos`)
2. Inspect the architecture
3. Run greedy and sampled generation
4. Visualise self-attention patterns

Source: [github.com/borisgraudt/mythos](https://github.com/borisgraudt/mythos)

## 1. Load the model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "bgraudt/mythos"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval();

## 2. Architecture summary

In [ ]:
cfg = model.config
n_params = sum(p.numel() for p in model.parameters()) / 1e6

print(f"Parameters:        {n_params:.1f} M")
print(f"Hidden layers:     {cfg.num_hidden_layers}")
print(f"Hidden size:       {cfg.hidden_size}")
print(f"Query / KV heads:  {cfg.num_attention_heads} / {cfg.num_key_value_heads}  (GQA)")
print(f"Vocabulary:        {cfg.vocab_size:,}")
print(f"Max context:       {cfg.max_position_embeddings}")

## 3. Generation

### Greedy

In [ ]:
prompt = "The history of artificial intelligence begins with"
inputs = tokenizer(prompt, return_tensors="pt")

out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

### Temperature / top-p sampling

In [ ]:
torch.manual_seed(42)
out = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 4. Attention visualisation

Hook into the first layer's attention to show how each token attends to the prior context.

In [ ]:
import matplotlib.pyplot as plt

text = "A transformer uses attention to relate tokens."
ids = tokenizer(text, return_tensors="pt").input_ids
tokens = [tokenizer.decode([t]) for t in ids[0].tolist()]

with torch.no_grad():
    out = model(ids, output_attentions=True)

# First layer, first head
attn = out.attentions[0][0, 0].detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn, cmap="viridis")
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)
ax.set_title("Layer 0 · Head 0 — attention weights")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 5. Next steps

- Full training guide: [`docs/TRAINING.md`](../docs/TRAINING.md)
- Architecture rationale: [`docs/RESEARCH.md`](../docs/RESEARCH.md)
- Hand-written attention, RoPE, SwiGLU: [`src/core/`](../src/core/)